# Предобработка MovieLens (ml-latest-small)

Загрузка → проверки → признаки фильмов → EDA → split по времени → аномальные пользователи (только по train) → индексы → сохранение.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

SEED = 42
DATA_DIR = Path("~/Workspace/ML_Project/data/row/ml-latest-small").expanduser()
OUT_DIR = Path("~/Workspace/ML_Project/data/processed").expanduser()
OUT_DIR.mkdir(parents=True, exist_ok=True)

VAL_FRAC = 0.1
TEST_FRAC = 0.1
DROP_ANOMALIES = False

sns.set_theme(style="whitegrid")

## 1. Загрузка и проверка целостности

In [ ]:
ratings = pd.read_csv(DATA_DIR / "ratings.csv")
movies = pd.read_csv(DATA_DIR / "movies.csv")
tags = pd.read_csv(DATA_DIR / "tags.csv")
links = pd.read_csv(DATA_DIR / "links.csv", dtype={"imdbId": str, "tmdbId": "Int64"})

print("shape:", ratings.shape, movies.shape, tags.shape, links.shape)
print("NaN:", {n: int(d.isna().sum().sum()) for n, d in
      [("ratings", ratings), ("movies", movies), ("tags", tags), ("links", links)]})
print("Дубликаты (userId, movieId):", ratings.duplicated(["userId", "movieId"]).sum())
print("Дубликаты movieId в movies:", movies.duplicated("movieId").sum())
print("movieId из ratings, которых нет в movies:", (~ratings["movieId"].isin(movies["movieId"])).sum())
print("Шкала оценок:", sorted(ratings["rating"].unique().tolist()))
n_u, n_m = ratings["userId"].nunique(), ratings["movieId"].nunique()
print(f"Пользователей: {n_u}, фильмов с оценками: {n_m}, разреженность: {1 - len(ratings) / (n_u * n_m):.2%}")

In [ ]:
ratings = (ratings.sort_values("timestamp")
                  .drop_duplicates(["userId", "movieId"], keep="last")
                  .reset_index(drop=True))
ratings["datetime"] = pd.to_datetime(ratings["timestamp"], unit="s")
print(ratings["datetime"].min(), "→", ratings["datetime"].max())

## 2. Фильмы: год, жанры, теги

In [ ]:
movies["title"] = movies["title"].str.strip()
year = movies["title"].str.extract(r"\((\d{4})\)\s*$")[0]
movies["year"] = pd.to_numeric(year, errors="coerce").astype("Int64")

years_fix = {
    "Babylon 5": 1993,
    "Ready Player One": 2018,
    "The Adventures of Sherlock Holmes and Doctor Watson": 1980,
    "Hyena Road": 2015,
    "Nocturnal Animals": 2016,
    "Paterson": 2016,
    "Moonlight": 2016,
    "The OA": 2016,
    "Cosmos": 2014,
    "Maria Bamford: Old Baby": 2017,
    "Death Note: Desu nôto (2006–2007)": 2006,
    "Generation Iron 2": 2017,
    "Black Mirror": 2013,
}
fix = pd.to_numeric(movies["title"].map(years_fix), errors="coerce").astype("Int64")
movies["year"] = movies["year"].fillna(fix)

print("Без года:", movies["year"].isna().sum(), movies.loc[movies["year"].isna(), "title"].tolist())
print("Диапазон лет:", movies["year"].min(), "–", movies["year"].max())

In [ ]:
movies["genres"] = movies["genres"].replace("(no genres listed)", "Unknown")
movies["genre_list"] = movies["genres"].str.split("|")
genres_all = sorted(movies["genre_list"].explode().unique())
for g in genres_all:
    movies[f"g_{g}"] = movies["genre_list"].apply(lambda x: int(g in x))
print(genres_all)
print("Фильмов без жанра:", (movies["genres"] == "Unknown").sum())

In [ ]:
tags = tags.dropna(subset=["tag"]).copy()
tags["tag"] = tags["tag"].astype(str).str.lower().str.strip()
tags = tags.drop_duplicates(["userId", "movieId", "tag"])

movie_tags = (tags.groupby("movieId")["tag"]
                  .agg(lambda s: "|".join(sorted(set(s)))).rename("tags"))
movies = movies.merge(movie_tags, on="movieId", how="left")
movies["tags"] = movies["tags"].fillna("")
print("Фильмов с тегами:", (movies["tags"] != "").sum(), "| уникальных тегов:", tags["tag"].nunique())

## 3. EDA

In [ ]:
r = ratings["rating"]
print(r.describe().round(3))
print("Мода:", r.mode().iloc[0], "| CV:", round(r.std() / r.mean() * 100, 1), "%")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(r, bins=np.arange(0.25, 5.75, 0.5), edgecolor="black")
ax[0].axvline(r.mean(), color="red", ls="--", label=f"среднее {r.mean():.2f}")
ax[0].axvline(r.median(), color="green", ls="--", label=f"медиана {r.median():.2f}")
ax[0].set_xticks(np.arange(0.5, 5.5, 0.5))
ax[0].set(title="Распределение оценок", xlabel="Оценка", ylabel="Количество")
ax[0].legend()

ratings.groupby(ratings["datetime"].dt.year).size().plot(kind="bar", ax=ax[1])
ax[1].set(title="Оценок по годам", xlabel="Год", ylabel="Количество")
plt.tight_layout()
plt.show()

In [ ]:
rg = ratings.merge(movies[["movieId", "genre_list"]], on="movieId").explode("genre_list")
genre_stats = (rg.groupby("genre_list")["rating"]
                 .agg(mean="mean", n="size").sort_values("mean"))
print(genre_stats.round(3))

plt.figure(figsize=(9, 6))
sns.barplot(x=genre_stats["mean"], y=genre_stats.index, color="steelblue")
plt.xlim(0, 5)
plt.title("Средняя оценка по жанрам")
plt.xlabel("Средняя оценка")
plt.ylabel("")
plt.show()

In [ ]:
movie_counts = ratings.groupby("movieId").size().sort_values(ascending=False)
user_counts = ratings.groupby("userId").size()
print(movie_counts.describe().round(1), "\n")
print(user_counts.describe().round(1), "\n")
print(f"Фильмов с < 10 оценками: {(movie_counts < 10).mean():.1%}")
print(f"Пользователей с < 20 оценками: {(user_counts < 20).mean():.1%}")
for n in [100, 500, 1000, 2000]:
    print(f"Топ-{n} фильмов: {movie_counts.head(n).sum() / movie_counts.sum():.1%} оценок")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(movie_counts.values)
ax[0].axhline(10, color="red", ls="--", label="порог 10")
ax[0].set(title="Длинный хвост", xlabel="Ранг фильма", ylabel="Число оценок")
ax[0].legend()
ax[1].loglog(np.arange(1, len(movie_counts) + 1), movie_counts.values)
ax[1].set(title="Log-log", xlabel="Ранг", ylabel="Число оценок")
plt.tight_layout()
plt.show()

## 4. Split по времени (на каждого пользователя: последние оценки → test, перед ними → val)

In [ ]:
ratings = ratings.sort_values(["userId", "timestamp", "movieId"]).reset_index(drop=True)
pos = ratings.groupby("userId").cumcount()
n_u = ratings.groupby("userId")["rating"].transform("size")
n_test = np.maximum(1, (n_u * TEST_FRAC).round()).astype(int)
n_val = np.maximum(1, (n_u * VAL_FRAC).round()).astype(int)

ratings["split"] = np.select(
    [pos >= n_u - n_test, pos >= n_u - n_test - n_val], ["test", "val"], default="train"
)
train = ratings[ratings["split"] == "train"]
print(ratings["split"].value_counts())
for s in ["val", "test"]:
    d = ratings[ratings["split"] == s]
    print(f"{s}: фильмов не из train — {(~d['movieId'].isin(train['movieId'])).mean():.1%}")

## 5. Аномальные пользователи (признаки считаются только по train)

In [ ]:
tr = train.merge(movies[["movieId", "genre_list"]], on="movieId")

g = tr.groupby("movieId")["rating"]
n_i, s_i = g.transform("size"), g.transform("sum")
loo_mean = ((s_i - tr["rating"]) / (n_i - 1)).where(n_i > 1)
tr["abs_dev"] = (tr["rating"] - loo_mean).abs()
tr["item_count"] = n_i
tr["extreme"] = ((tr["rating"] <= 1.0) | (tr["rating"] >= 4.5)).astype(float)

uf = tr.groupby("userId").agg(
    n_ratings=("rating", "size"),
    mean_rating=("rating", "mean"),
    std_rating=("rating", "std"),
    frac_extreme=("extreme", "mean"),
    mean_popularity=("item_count", "mean"),
    mean_abs_dev=("abs_dev", "mean"),
)

ge = tr[["userId", "genre_list"]].explode("genre_list")
c = ge.groupby(["userId", "genre_list"]).size()
p = c / c.groupby(level=0).transform("sum")
uf["genre_entropy"] = -(p * np.log(p)).groupby(level=0).sum()

uf["log_n_ratings"] = np.log1p(uf.pop("n_ratings"))
uf["log_popularity"] = np.log1p(uf.pop("mean_popularity"))
uf["std_rating"] = uf["std_rating"].fillna(0)
uf["mean_abs_dev"] = uf["mean_abs_dev"].fillna(uf["mean_abs_dev"].median())

feature_cols = list(uf.columns)
print(uf.shape)
print(uf.describe().round(2))

In [ ]:
corr = uf[feature_cols].corr(method="spearman")
plt.figure(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5)
plt.title("Корреляции признаков пользователей (Spearman)")
plt.tight_layout()
plt.show()

pairs = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
print("Пары с |corr| > 0.7:")
print(pairs[pairs.abs() > 0.7].sort_values(ascending=False))

In [ ]:
X = StandardScaler().fit_transform(uf[feature_cols])

iso = IsolationForest(n_estimators=300, contamination="auto", random_state=SEED, n_jobs=-1).fit(X)
lof = LocalOutlierFactor(n_neighbors=20, contamination="auto", n_jobs=-1)
lof_pred = lof.fit_predict(X)

uf["iso_score"] = iso.decision_function(X)
uf["lof_score"] = lof.negative_outlier_factor_
uf["anomaly_iso"] = iso.predict(X) == -1
uf["anomaly_lof"] = lof_pred == -1
uf["suspect"] = uf["anomaly_iso"] & uf["anomaly_lof"]

print(f"IF: {uf['anomaly_iso'].sum()} | LOF: {uf['anomaly_lof'].sum()} | оба: {uf['suspect'].sum()} из {len(uf)}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(uf["iso_score"], bins=60, color="steelblue")
ax[0].axvline(0, color="red", ls="--", label="граница (score = 0)")
ax[0].set(title="Isolation Forest", xlabel="decision_function (ниже = аномальнее)")
ax[0].legend()
ax[1].hist(uf["lof_score"], bins=60, color="steelblue")
ax[1].axvline(lof.offset_, color="red", ls="--", label=f"граница ({lof.offset_:.1f})")
ax[1].set(title="LOF", xlabel="negative_outlier_factor_ (ниже = аномальнее)")
ax[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
show = uf[uf["anomaly_iso"] | uf["anomaly_lof"]].copy()
show["n_ratings"] = np.expm1(show["log_n_ratings"]).round().astype(int)
cols = ["n_ratings", "mean_rating", "std_rating", "frac_extreme", "mean_abs_dev",
        "genre_entropy", "iso_score", "lof_score", "suspect"]
print(show.sort_values("lof_score")[cols].head(20).round(2).to_string())

In [ ]:
pca = PCA(n_components=2, random_state=SEED)
Z = pca.fit_transform(X)

plt.figure(figsize=(9, 6))
norm = ~(uf["anomaly_iso"] | uf["anomaly_lof"]).values
plt.scatter(Z[norm, 0], Z[norm, 1], s=12, alpha=0.4, label="норма")
only = (uf["anomaly_iso"] ^ uf["anomaly_lof"]).values
plt.scatter(Z[only, 0], Z[only, 1], s=40, c="orange", label="один метод")
both = uf["suspect"].values
plt.scatter(Z[both, 0], Z[both, 1], s=60, c="red", marker="x", label="оба метода")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
plt.title("Пользователи в PCA-проекции")
plt.legend()
plt.show()

print("Объяснено дисперсии:", f"{pca.explained_variance_ratio_.sum():.1%}")
print(pd.DataFrame(pca.components_, columns=feature_cols, index=["PC1", "PC2"]).round(2).T)

## 6. Итоговые выборки, индексы, сохранение

In [ ]:
suspects = uf.index[uf["suspect"]]
final = ratings[~ratings["userId"].isin(suspects)] if DROP_ANOMALIES else ratings
print(f"Подозрительных пользователей: {len(suspects)} | удалены: {DROP_ANOMALIES} | оценок в итоге: {len(final)}")

uid_map = {u: i for i, u in enumerate(sorted(final["userId"].unique()))}
iid_map = {m: i for i, m in enumerate(sorted(movies["movieId"].unique()))}

final = final.assign(uid=final["userId"].map(uid_map), iid=final["movieId"].map(iid_map))
cols = ["userId", "movieId", "uid", "iid", "rating", "timestamp"]
for s in ["train", "val", "test"]:
    final.loc[final["split"] == s, cols].to_csv(OUT_DIR / f"{s}.csv", index=False)

movies_out = (movies.drop(columns=["genre_list"])
                    .assign(iid=movies["movieId"].map(iid_map))
                    .merge(links, on="movieId", how="left"))
movies_out.to_csv(OUT_DIR / "movies.csv", index=False)

users_out = uf.reset_index().assign(uid=lambda d: d["userId"].map(uid_map))
users_out.to_csv(OUT_DIR / "users.csv", index=False)

print(sorted(p.name for p in OUT_DIR.iterdir()))
print({s: int((final["split"] == s).sum()) for s in ["train", "val", "test"]})